# Dunnhumby: The Complete Journey | 01 Data Download

Downloads the Kaggle dataset `frtgnn/dunnhumby-the-complete-journey` (8 CSV files, about 847 MB) into this project's Drive folder, converts each table to Parquet, and runs basic checks.

**Folder layout created in `MyDrive/Dunnhumby Project`:**
- `data/raw/` original CSV files
- `data/parquet/` same tables as Parquet (much smaller and faster to query)

**One-time setup (Kaggle API token):**
1. Go to kaggle.com, then Settings, then API, and generate a new **API token** (it starts with `KGAT_`). Copy it, because Kaggle shows it only once.
2. In Colab, click the **key icon (Secrets)** in the left sidebar.
3. Add one secret named `KAGGLE_API_TOKEN` with the token as its value, and turn on **Notebook access**.

The notebook skips steps already done, so it is safe to run again.

In [1]:
!pip -q install -U kaggle duckdb
from google.colab import drive, userdata
from pathlib import Path
import os, duckdb, pandas as pd
drive.mount('/content/drive')
PROJ = Path('/content/drive/MyDrive/Dunnhumby Project')
RAW, PQ = PROJ/'data/raw', PROJ/'data/parquet'
for p in (RAW, PQ): p.mkdir(parents=True, exist_ok=True)
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 53.9 MB/s eta 0:00:00
Mounted at /content/drive


## Download and unzip
Downloads to Colab's local disk first (faster), then copies the CSVs to Drive.

In [2]:
if len(list(RAW.glob('*.csv'))) < 8:
    !kaggle datasets download -d frtgnn/dunnhumby-the-complete-journey -p /content/dh --unzip -q
    !cp /content/dh/*.csv "{RAW}"
print(*sorted(f'{f.name}: {f.stat().st_size/1e6:.1f} MB' for f in RAW.glob('*.csv')), sep='\n')

Dataset URL: https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey
License(s): DbCL-1.0
campaign_desc.csv: 0.0 MB
campaign_table.csv: 0.1 MB
causal_data.csv: 695.9 MB
coupon.csv: 2.8 MB
coupon_redempt.csv: 0.1 MB
hh_demographic.csv: 0.0 MB
product.csv: 6.4 MB
transaction_data.csv: 141.7 MB


## Convert to Parquet
`causal_data.csv` is the largest file and takes the longest.

In [3]:
con = duckdb.connect()
for f in sorted(RAW.glob('*.csv')):
    if not (out := PQ/f'{f.stem}.parquet').exists(): con.execute(f"COPY (SELECT * FROM read_csv_auto('{f}')) TO '{out}' (FORMAT PARQUET)"); print('done', f.stem)

done campaign_desc
done campaign_table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done causal_data
done coupon
done coupon_redempt
done hh_demographic
done product


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done transaction_data


## Check tables
Row count, column count and Parquet size for each table, then the first 3 rows of each.

In [4]:
t = lambda n: f"'{PQ}/{n}.parquet'"
pd.DataFrame([(f.stem, con.execute(f"SELECT COUNT(*) FROM {t(f.stem)}").fetchone()[0], len(con.execute(f"SELECT * FROM {t(f.stem)} LIMIT 0").description), round(f.stat().st_size/1e6, 1)) for f in sorted(PQ.glob('*.parquet'))], columns=['table', 'rows', 'cols', 'parquet_MB'])

,table,rows,cols,parquet_MB
0,campaign_desc,30,4,0.0
1,campaign_table,7208,3,0.0
2,causal_data,36786524,5,46.5
3,coupon,124548,3,0.7
4,coupon_redempt,2318,4,0.0
5,hh_demographic,801,8,0.0
6,product,92353,7,1.1
7,transaction_data,2595732,12,24.0


In [5]:
for f in sorted(PQ.glob('*.parquet')): print(f'--- {f.stem}', con.execute(f"SELECT * FROM {t(f.stem)} LIMIT 3").df().to_string(), sep='\n', end='\n\n')

--- campaign_desc
  DESCRIPTION  CAMPAIGN  START_DAY  END_DAY
0       TypeB        24        659      719
1       TypeC        15        547      708
2       TypeB        25        659      691

--- campaign_table
  DESCRIPTION  household_key  CAMPAIGN
0       TypeA             17        26
1       TypeA             27        26
2       TypeA            212        26

--- causal_data
   PRODUCT_ID  STORE_ID  WEEK_NO display mailer
0       26190       286       70       0      A
1       26190       288       70       0      A
2       26190       289       70       0      A

--- coupon
    COUPON_UPC  PRODUCT_ID  CAMPAIGN
0  10000089061       27160         4
1  10000089064       27754         9
2  10000089073       28897        12

--- coupon_redempt
   household_key  DAY   COUPON_UPC  CAMPAIGN
0              1  421  10000085364         8
1              1  421  51700010076         8
2              1  427  54200000033         8

--- hh_demographic
  AGE_DESC MARITAL_STATUS_CODE INCOME_DES

## Quick sanity checks
- How many households have demographics (expect roughly 800 of 2,500)
- Time span: `DAY` and `WEEK_NO` are day and week numbers, not calendar dates

In [6]:
con.execute(f"""SELECT COUNT(DISTINCT tr.household_key) hh_total, COUNT(DISTINCT d.household_key) hh_with_demo,
MIN(tr.DAY) first_day, MAX(tr.DAY) last_day, MIN(tr.WEEK_NO) first_week, MAX(tr.WEEK_NO) last_week, ROUND(SUM(tr.SALES_VALUE)) total_sales
FROM {t('transaction_data')} tr LEFT JOIN {t('hh_demographic')} d USING (household_key)""").df()

,hh_total,hh_with_demo,first_day,last_day,first_week,last_week,total_sales
0,2500,801,1,711,1,102,8057463.0
